# NB_bishop_ch16_pca_latent_variables

**Chapter 16 — PCA and Latent Variable Models**

This notebook implements PCA from scratch via eigendecomposition, applies it to MNIST and face datasets, introduces probabilistic PCA, and compares PCA with autoencoder-based dimensionality reduction.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_16/NB_bishop_ch16_pca_latent_variables.ipynb)

In [ ]:
# Install dependencies (Colab-safe)
# !pip install tensorflow numpy matplotlib scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.datasets import fetch_olivetti_faces, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as skPCA
import tensorflow as tf

print(f'NumPy version: {np.__version__}')
print(f'TensorFlow version: {tf.__version__}')

In [ ]:
# Chart styling setup
matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

## 1. PCA from Scratch — Eigendecomposition

In [ ]:
# Load MNIST via Keras
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
X_flat = X_train.reshape(-1, 784).astype(np.float64) / 255.0
print(f'MNIST training set: {X_flat.shape}')

In [ ]:
# PCA from scratch via eigendecomposition
def pca_eigen(X, n_components):
    """PCA using eigendecomposition of the covariance matrix."""
    # Center data
    mean = np.mean(X, axis=0)
    X_centered = X - mean
    
    # Covariance matrix
    cov = np.cov(X_centered, rowvar=False)
    
    # Eigendecomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    
    # Sort by descending eigenvalue
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # Select top components
    W = eigenvectors[:, :n_components]
    Z = X_centered @ W
    
    return Z, W, eigenvalues, mean

print('PCA function defined.')

In [ ]:
# Apply PCA to a subset of MNIST (for speed)
np.random.seed(42)
subset_idx = np.random.choice(len(X_flat), 10000, replace=False)
X_sub = X_flat[subset_idx]
y_sub = y_train[subset_idx]

n_components = 50
Z, W, eigenvalues, data_mean = pca_eigen(X_sub, n_components)
print(f'Projected shape: {Z.shape}')
print(f'Top 10 eigenvalues: {eigenvalues[:10].round(4)}')

In [ ]:
# Explained variance ratio
explained_var = eigenvalues / eigenvalues.sum()
cumulative_var = np.cumsum(explained_var)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 51), explained_var[:50], color='#1a3a6e', alpha=0.8)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot (Top 50 Components)')

axes[1].plot(range(1, len(cumulative_var)+1), cumulative_var, color='#cd0000', linewidth=2)
axes[1].axhline(0.95, color='gray', linestyle='--', alpha=0.7, label='95% threshold')
n95 = np.argmax(cumulative_var >= 0.95) + 1
axes[1].axvline(n95, color='#228B22', linestyle='--', alpha=0.7, label=f'{n95} components')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

fig.tight_layout()
save_fig(fig, 'bishop_ch16_pca_eigenvalues')
plt.show()
print(f'Components needed for 95% variance: {n95}')

## 2. PCA on MNIST — Visualization

In [ ]:
# 2D projection of MNIST
Z2d, _, _, _ = pca_eigen(X_sub, 2)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.tab10(np.linspace(0, 1, 10))
for digit in range(10):
    mask = y_sub == digit
    ax.scatter(Z2d[mask, 0], Z2d[mask, 1], c=[colors[digit]], s=5, alpha=0.5, label=str(digit))
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('MNIST — 2D PCA Projection')
ax.legend(markerscale=4)
fig.tight_layout()
plt.show()

In [ ]:
# Visualize top eigenvectors (eigenfaces/eigendigits)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(W[:, i].reshape(28, 28), cmap='RdBu_r')
    ax.set_title(f'PC {i+1}')
    ax.axis('off')
fig.suptitle('Top 10 Principal Components (MNIST)', fontsize=14)
fig.tight_layout()
plt.show()

## 3. Reconstruction with Varying Components

In [ ]:
# Reconstruct images with different numbers of components
sample_idx = [0, 1, 2, 3, 4]
n_list = [5, 10, 20, 50, 100, 200]

fig, axes = plt.subplots(len(sample_idx), len(n_list) + 1, figsize=(16, 8))

for row, si in enumerate(sample_idx):
    axes[row, 0].imshow(X_sub[si].reshape(28, 28), cmap='gray')
    axes[row, 0].set_title('Original' if row == 0 else '')
    axes[row, 0].axis('off')
    
    for col, nc in enumerate(n_list):
        Z_nc, W_nc, _, mean_nc = pca_eigen(X_sub, nc)
        X_recon = Z_nc @ W_nc.T + mean_nc
        axes[row, col+1].imshow(X_recon[si].reshape(28, 28), cmap='gray')
        axes[row, col+1].set_title(f'k={nc}' if row == 0 else '')
        axes[row, col+1].axis('off')

fig.suptitle('PCA Reconstruction with Varying Components', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch16_pca_reconstruction')
plt.show()

In [ ]:
# Reconstruction error vs number of components
errors = []
comp_range = [2, 5, 10, 20, 50, 100, 150, 200, 300, 400, 500]
for nc in comp_range:
    Z_nc, W_nc, _, mean_nc = pca_eigen(X_sub, nc)
    X_recon = Z_nc @ W_nc.T + mean_nc
    mse = np.mean((X_sub - X_recon)**2)
    errors.append(mse)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(comp_range, errors, 'o-', color='#1a3a6e', linewidth=2, markersize=6)
ax.set_xlabel('Number of Components')
ax.set_ylabel('Mean Squared Error')
ax.set_title('Reconstruction Error vs Components')
fig.tight_layout()
plt.show()

## 4. PCA on Face Dataset

In [ ]:
# Load Olivetti faces
faces = fetch_olivetti_faces()
X_faces = faces.data  # (400, 4096) — 64x64 images
y_faces = faces.target
print(f'Faces dataset: {X_faces.shape}, {len(np.unique(y_faces))} subjects')

In [ ]:
# PCA on faces — eigenfaces
Z_faces, W_faces, evals_faces, mean_face = pca_eigen(X_faces, 50)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes[0, 0].imshow(mean_face.reshape(64, 64), cmap='gray')
axes[0, 0].set_title('Mean Face')
axes[0, 0].axis('off')
for i in range(1, 10):
    r, c = divmod(i, 5)
    axes[r, c].imshow(W_faces[:, i-1].reshape(64, 64), cmap='RdBu_r')
    axes[r, c].set_title(f'Eigenface {i}')
    axes[r, c].axis('off')
fig.suptitle('Mean Face and Top 9 Eigenfaces', fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Face reconstruction with varying components
face_idx = [0, 40, 100, 200, 350]
face_n_list = [5, 10, 25, 50, 100, 200]

fig, axes = plt.subplots(len(face_idx), len(face_n_list) + 1, figsize=(16, 10))
for row, fi in enumerate(face_idx):
    axes[row, 0].imshow(X_faces[fi].reshape(64, 64), cmap='gray')
    axes[row, 0].set_title('Original' if row == 0 else '')
    axes[row, 0].axis('off')
    for col, nc in enumerate(face_n_list):
        Zf, Wf, _, mf = pca_eigen(X_faces, nc)
        Xr = Zf @ Wf.T + mf
        axes[row, col+1].imshow(np.clip(Xr[fi].reshape(64, 64), 0, 1), cmap='gray')
        axes[row, col+1].set_title(f'k={nc}' if row == 0 else '')
        axes[row, col+1].axis('off')
fig.suptitle('Face Reconstruction with Varying Components', fontsize=14)
fig.tight_layout()
plt.show()

## 5. Probabilistic PCA (PPCA)

In [ ]:
# Probabilistic PCA implementation (EM algorithm)
def ppca_em(X, q, max_iter=100, tol=1e-6):
    """Probabilistic PCA via EM algorithm.
    
    X: (N, D) data matrix
    q: latent dimensionality
    Returns: W (D, q), sigma2, Z (N, q)
    """
    N, D = X.shape
    mu = X.mean(axis=0)
    X_c = X - mu
    
    # Initialize
    W = np.random.randn(D, q) * 0.01
    sigma2 = 1.0
    
    for iteration in range(max_iter):
        # E-step: compute expected latent variables
        M = W.T @ W + sigma2 * np.eye(q)  # (q, q)
        M_inv = np.linalg.inv(M)
        Z = X_c @ W @ M_inv.T  # (N, q)
        
        # E[z z^T]
        Ez_zt = sigma2 * M_inv + Z.T @ Z / N  # (q, q)
        
        # M-step
        W_new = (X_c.T @ Z) @ np.linalg.inv(N * Ez_zt)  # (D, q)
        
        sigma2_new = (np.sum(X_c**2) - 2 * np.trace(Z.T @ X_c @ W_new)
                      + np.trace(Ez_zt @ W_new.T @ W_new) * N) / (N * D)
        
        # Check convergence
        dW = np.linalg.norm(W_new - W)
        W = W_new
        sigma2 = max(sigma2_new, 1e-10)
        
        if dW < tol:
            print(f'PPCA converged at iteration {iteration+1}')
            break
    
    return W, sigma2, Z, mu

print('PPCA EM function defined.')

In [ ]:
# Apply PPCA to MNIST subset
np.random.seed(42)
X_ppca = X_sub[:2000]  # smaller subset for speed

W_ppca, sigma2, Z_ppca, mu_ppca = ppca_em(X_ppca, q=2, max_iter=200)
print(f'Estimated noise variance: {sigma2:.6f}')
print(f'Latent projections shape: {Z_ppca.shape}')

In [ ]:
# Compare PCA vs PPCA 2D projections
Z_pca_2d, _, _, _ = pca_eigen(X_ppca, 2)
y_ppca = y_sub[:2000]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = plt.cm.tab10(np.linspace(0, 1, 10))

for digit in range(10):
    mask = y_ppca == digit
    axes[0].scatter(Z_pca_2d[mask, 0], Z_pca_2d[mask, 1], c=[colors[digit]], s=8, alpha=0.5, label=str(digit))
    axes[1].scatter(Z_ppca[mask, 0], Z_ppca[mask, 1], c=[colors[digit]], s=8, alpha=0.5, label=str(digit))

axes[0].set_title('Standard PCA (2D)')
axes[0].set_xlabel('PC 1'); axes[0].set_ylabel('PC 2')
axes[0].legend(markerscale=3, fontsize=8)
axes[1].set_title('Probabilistic PCA (2D)')
axes[1].set_xlabel('Latent 1'); axes[1].set_ylabel('Latent 2')
axes[1].legend(markerscale=3, fontsize=8)

fig.suptitle('PCA vs Probabilistic PCA on MNIST', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch16_ppca')
plt.show()

## 6. PPCA — Automatic Dimensionality Selection

In [ ]:
# Log-likelihood for PPCA as function of q
def ppca_log_likelihood(X, q):
    """Compute approximate log-likelihood for PPCA model."""
    N, D = X.shape
    mu = X.mean(axis=0)
    X_c = X - mu
    S = X_c.T @ X_c / N  # sample covariance
    
    # Eigendecomposition of S
    evals, _ = np.linalg.eigh(S)
    evals = np.sort(evals)[::-1]
    
    # MLE for sigma2
    sigma2 = np.mean(evals[q:])
    sigma2 = max(sigma2, 1e-10)
    
    # Log-likelihood (Bishop eq 12.45 approximation)
    ll = -N * D / 2 * np.log(2 * np.pi)
    ll -= N / 2 * (np.sum(np.log(evals[:q] + 1e-10)) + (D - q) * np.log(sigma2))
    ll -= N * D / 2
    
    return ll

q_range = range(1, 50)
lls = [ppca_log_likelihood(X_ppca, q) for q in q_range]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(q_range), lls, 'o-', color='#1a3a6e', markersize=4, linewidth=1.5)
ax.set_xlabel('Latent Dimensionality (q)')
ax.set_ylabel('Log-Likelihood')
ax.set_title('PPCA Model Selection')
fig.tight_layout()
plt.show()

## 7. Comparison: PCA vs Autoencoder

In [ ]:
# Build a simple autoencoder for comparison
latent_dim = 20

encoder = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(784,)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(latent_dim, activation='linear', name='latent')
])

decoder = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(latent_dim,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(784, activation='sigmoid')
])

autoencoder_input = tf.keras.Input(shape=(784,))
encoded = encoder(autoencoder_input)
decoded = decoder(encoded)
autoencoder = tf.keras.Model(autoencoder_input, decoded)

autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

In [ ]:
# Train autoencoder
X_ae_train = X_flat[:50000]
X_ae_val = X_flat[50000:]

history_ae = autoencoder.fit(
    X_ae_train, X_ae_train,
    epochs=30,
    batch_size=256,
    validation_data=(X_ae_val, X_ae_val),
    verbose=1
)
print(f'Final AE val loss: {history_ae.history["val_loss"][-1]:.6f}')

In [ ]:
# Compare reconstructions: PCA (20 components) vs Autoencoder (20-dim latent)
X_test_flat = X_test.reshape(-1, 784).astype(np.float64) / 255.0
test_samples = X_test_flat[:8]

# PCA reconstruction
Z_test_pca, W_test, _, mean_test = pca_eigen(X_flat, 20)
test_centered = test_samples - mean_test
test_proj = test_centered @ W_test
pca_recon = test_proj @ W_test.T + mean_test

# AE reconstruction
ae_recon = autoencoder.predict(test_samples.astype(np.float32), verbose=0)

fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for i in range(8):
    axes[0, i].imshow(test_samples[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title('Original' if i == 0 else '')
    
    axes[1, i].imshow(np.clip(pca_recon[i].reshape(28, 28), 0, 1), cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title('PCA (k=20)' if i == 0 else '')
    
    axes[2, i].imshow(ae_recon[i].reshape(28, 28), cmap='gray')
    axes[2, i].axis('off')
    axes[2, i].set_title('Autoencoder' if i == 0 else '')

fig.suptitle('PCA vs Autoencoder Reconstruction (latent dim = 20)', fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison: MSE vs latent dim
dims = [2, 5, 10, 20, 50]
pca_mses = []

for d in dims:
    Zd, Wd, _, md = pca_eigen(X_flat[:5000], d)
    Xr = Zd @ Wd.T + md
    pca_mses.append(np.mean((X_flat[:5000] - Xr)**2))

print('PCA reconstruction MSE by dimension:')
for d, mse in zip(dims, pca_mses):
    print(f'  dim={d}: MSE={mse:.6f}')
print(f'\nAutoencoder (dim=20) val MSE: {history_ae.history["val_loss"][-1]:.6f}')

## Summary

In this notebook we covered:
- **PCA from scratch** via eigendecomposition of the covariance matrix
- **Scree plot and cumulative variance** for selecting the number of components
- **Eigenfaces** — applying PCA to the Olivetti faces dataset
- **Probabilistic PCA** — a generative model perspective via EM
- **PCA vs Autoencoder** — the nonlinear autoencoder achieves lower reconstruction error for the same latent dimensionality

In [ ]:
# Key takeaways
takeaways = [
    '1. PCA finds orthogonal directions of maximum variance via eigendecomposition.',
    '2. The scree plot and cumulative variance help select the number of components.',
    '3. Probabilistic PCA adds a noise model and enables missing data imputation.',
    '4. Eigenfaces demonstrate PCA on high-dimensional image data.',
    '5. Nonlinear autoencoders outperform linear PCA for complex data manifolds.'
]
print('Key Takeaways:')
print('-' * 60)
for t in takeaways:
    print(t)